In [1]:
from pathlib import Path 
from subprocess import run
import os
import sys
import copykitten

# Authenticating with SSH keys

You have probably heard that SSH keys are a secure way to connect to a remote server. If you've ever used them, you know that they can also be very convenient. 

Secure and convenient? What's not to like? 

But they do have a downside. To get ready to use them, most people run a series of commands they don't understand and dread the thought of having to do it again. 

This notebook can help in two ways.  

- It explains each step. 

- It makes it easy to repeat the process. 

## The Joke You Should Remember

> Asymmetric cryptography is a system for turning every other security problem into a key management problem.

The joke follows a familiar pattern: create expectations, then disappoint. It's not very funny, but people who understand computer security like it because it is so true.

Understanding the phrase &ldquo;key management problem&rdquo; is the key to understanding the many steps that are outlined in most explanations about how to set up ssh-keys. 

- The basic steps solve the authentication problem for client connecting with a server. 
- The other steps grapple with the key management problem that the solution to the authentication problem creates. 

## The Fact You Should Remember 

There is no solution to the key management problem. The best you can hope for is a trade-off between convenience and security. 


## I. Solving the Authentication Problem 

With symmetric cryptography, both sides have access to secret. Asymmetric cryptography relies on a pair of keys: a secret key and a public key that the client creates and controls. The asymmetry arises because the server only sees the public key. 

When the server has your public key, you can securely authenticate with the server. But how do you know that you are connecting to the server you want to connect with. You need to authenticate the server, make it prove that it really is controlled by Github. Now the roles are reversed. Github needs its own key pair. You need to have a copy of its public key. 

**The Four Steps That Solve the Authentication Problem**
1. Create your key-pair
2. Tell the operating system to use that keypair when you want to connect to github.com
3. Through an independent channel, save your public key on Github's servers.
4. Through an independent channel, get a copy of Github's public key. 

### Step 1: Create your key pair
For SSH authentication, you need a key pair. Before creating it, you must provide a name. 

This command provides a default `ed25519` that refers to the default algorithm that SSH uses but it is just name. You can change it to any other string before you run the code in the cell. 

In [19]:
key_name = "id_ed25519"

If you use this name, you'll end up with two new files in your `.ssh` folder. One will have no file extension. It has your secret key. The other will have the file extension ".pub". It has your public key. 

The names tell you something important. You can reveal your public key to anyone, including github. This poses no security threat. But if someone learns your secret key, the security of this system is completely broken. They can impersonate you. 

The code in next cell checks whether you already have a key pair that uses the value you selected for `key_name`.

In [20]:
ssh = Path.home() / ".ssh" 
secret_key_file = ssh / key_name 
public_key_file = ssh / (key_name + ".pub")
if secret_key_file.exists() or public_key_file.exists():
    print(f"Try a different value for key_name.")
    print(f"You already have a key with the name {key_name}.")


If you already have a key that uses `key_name`, you can delete the existing keys if you aren't using them. If you aren't sure, it is safer to pick a different value for `key_name` and rerun these two cells.

The next cell runs a command that accepts a variable `key_path_string`. It is a string that tells SSH where to save your secret key. 

The command also accepts a variable `passphrase` that for now is set to an empty string. This means that you will create a secret ssh key that is not protected by a passphrase. My strong suggestion is to make sure that you can authenticate with Github first without using a passphrase. Then, work through the companion notebook `protecting_ssh.ipynb` before making any decisions about how best to protect you SSH setup. 

In [21]:
string_key_path = str(secret_key_file)
passphrase = ""

if secret_key_file.exists():
    print("Change key_name and try again")
    print(f"The file {string_key_path} already exists.") 
else:
    cp = run(
                [
                    "ssh-keygen", 
                    "-f", 
                    string_key_path, 
                    "-N", 
                    passphrase,
                    "-C",
                    key_name,
                    "-q",
                ]
            )
    if cp.returncode == 0:
        print(f"Keys were created in {str(ssh)}.")
    

Keys were created in /Users/pr/.ssh.


If you keys were created, the function will say so. But it never hurts to check: 

In [22]:
print(f"{ssh / key_name} exists? {(ssh / key_name).exists()}")
print(f"{ssh / (key_name + ".pub")} exists? {(ssh / key_name).exists()}")

/Users/pr/.ssh/id_ed25519 exists? True
/Users/pr/.ssh/id_ed25519.pub exists? True


### Step 2: Tell Git which key pair to use for Github 

If you use a git command such as `git push` or `git pull`, from a folder that you downloaded from Github, git will have a record of the server that it should pull from or push to. But git will not know which keypair it should use to authenticate with Github. You need to tell git which keypair to use. 

The simplest way to do that is to have a file called `config` in your `.ssh` directory and to put an entry in that file that looks like this: 

In [23]:
print(f"\nHost github.com\n  IdentityFile {ssh / key_name}\n")


Host github.com
  IdentityFile /Users/pr/.ssh/id_ed25519



If you don't have a `config` file, the function in the next cell will create one and add this entry. 

If you already have a `config` file, the function will make a backup called `config.back`, then 
- if you don't have an entry for Github, it will add one.
- if you have a different entry for Github, it will replace it with this one.


In [24]:
def update_config(key_name: str):
    """Update the config file in the .ssh folder to include 
    an entry for github using the new key specified by key_name.
    """
    config = (Path.home() / ".ssh/config")
    new_entry = ["Host github.com", f"  IdentityFile {ssh / key_name}", ""] 
    
    if not config.exists():
        config = new_entry
        config.write_text("\n".join(new_entry))
    
    else:
        (config.parent / "config.back").write_text(config.read_text())
        new_cfg = []
        for line in config.read_text().splitlines():
            new_cfg.append(line.rstrip())
        if "Host github.com" in new_cfg:
            start_github_entry = new_cfg.index("Host github.com")
            try:
                start_next_entry = new_cfg.index("", start_github_entry) + 1
            except ValueError:
                start_next_entry = len(new_cfg)
            new_cfg[start_github_entry:start_next_entry] = new_entry
    
        else:
            new_cfg = new_cfg.extend(new_entry) 
            
    if not new_cfg[-1] == "":
        new_cfg.append("")
        
    s = "\n".join(new_cfg) 
    m = config.write_text(s)

    print("You now have a config file with this entry for github.com:")
    print(f"\nHost github.com\n  IdentityFile {ssh / key_name}\n")
    return

update_config(key_name)


You now have a config file with this entry for github.com:

Host github.com
  IdentityFile /Users/pr/.ssh/id_ed25519



### Step 3: Save your public key with Github

Now we begin to see what key management involves. Before you can make an SSH connection to github.com, you need to have your public key stored there. So you have to have some other way to authenticate with github.com and put your public key there. 

Your pulic key is stored in this file:

In [25]:
print(ssh / (key_name + ".pub"))


/Users/pr/.ssh/id_ed25519.pub


Then have to get the contents from that file and paste them into a window on Github's website. 

To get ready, open a browser in another window on your computer. Log into Github.com. If you don't already have an account with Github, you'll have to register. Once you are registered, use your browser to login. 

When you login, you land on the Home page. In the upper right corner, you'll see an icon that represents you. Click on it and scan the list that opens below. About 2/3rds of way down, you'll find the link`Settings`. After you click on that link, you'll land on the Settings page. 

On the Settings page, look down the list on the left side for the entry "SSH and GPG" and click on it. 

On the keys page where you land, there is a green button in the upper right that says "New SSH key". Locate it, but don't click it yet. Instead, come back to this notebook and run the next cell. It uses the copy_kitten library to put your public key onto the system clipboard. This makes it easy for you to do the required paste on Github. By relying on code, you can be sure not to make the common mistake of copying the secret key to the clipboard! 

In [26]:
def publickey_to_clipboard(key_name):
    """Use copykitten.copy() to put the public key on the clipboard"""
    p = Path.home() / (".ssh/" + key_name + ".pub")
    s = p.read_text()
    copykitten.copy(s)
    return

publickey_to_clipboard(key_name)


Now, switch back to your browser. On the keys page, click on "New SSH key". You'll land on a new key page with a title window, a type selector, and a key window. 

Click in the key window and use the paste keyboard-shortcut (CMD v on macOS or CTRL v on Windows). Your key will be pasted into the window. 

All that's left is to confirm by clicking on the green "Add Key" button below the key window. 

You don't need to supply a name for the key. Github will read it from the comment at the end of your key. 

Leave the type selector in its default state, "Authentication Key." 

## Step 4: Get Github's public key 

At this point, you would be able to authenticate with the server. You have the local copy of your secret key. The server has your public key. 

But how do you know that you are connecting to the right server? You should ask the server to authenticate with you. 

The risk that you connect to the wrong server is low but it isn't zero. When you tell SSH to make a connection with github.com, SSH relies on the DNS system to find the IP address for github.com and then connects with that IP address. There are known attacks that can trick the DNS system into returning the wrong IP address, one that could point to a server controlled by bad actors. When you tried to log into Github.com, this same type of attack may have tricked your browser into using the same incorrect IP address. There are other protections for the HTTPS protocol that your browser relies on, but the most secure way to use SSH is to make sure that you have a copy of the public key for the SSH server at Github so that SSH can use it to authenticate the server after it makes the connection. 

In practice, the risk that SSH connects to the wrong server is low. Almost everyone ignores it when they sign into a server the first time. By default, they follow a strategy known as "trust on first use." The first time you make an SSH connection to a server, you trust that you are connecting to the right one. When you connect, the server sends you its public key, which is saved on your computer. On all subsequent connections, you can confirm that you are connecting to the same server as the first time by using that public key to verify a signed message that the server sends you. 

The only indication about what is going on with "trust on first use" is a message that SSH shows after the first connection to a server. This next lines are from a terminal session in which I asked ssh to make a connection to git@github.com. 

```
$ ssh -T git@github.com
```

Here is the message I got back: 

```
The authenticity of host 'github.com (140.82.113.4)' can't be established.
ED25519 key fingerprint is: SHA256:+DiY3wvvV6TuJJhbpZisF/zLDA0zPMSvHdkr4UvCOqU
This key is not known by any other names.
Are you sure you want to continue connecting (yes/no/[fingerprint])? 
```

What most people do is answer yes. If I had responded with &lsquo;yes&rsquo; the connection would have gone through and SSH would have received the public key that github.com uses for the Ed25519 protocol and saved it in the `known_hosts` file in my `.ssh` directory. 

But to drive home the principle that public keys need to be communicated through an independent channel before you can make a secure connection, the approach here will be different. I looked up the public key in the Github documentation. (If you want to check for yourself, the URL is in the doc string for the function in the next cell.) Here is what Github publishes there there: 

<dir>
    <img src="./github_keys.png">
</dir>

The first entry (which is a single line but spills over in the display here) has the public key that Github.com uses for the Ed25519 protocol.  

The code in the next cell creates a `known_hosts` file if you don't have one and makes sure that you have Github's Ed25519 key in that file. 

In [16]:
def check_known_hosts(s: str = ""):
    """Checks the known hosts file for a line that contains the string s. 
    If no string is supplied, the function uses the ed25519 entry
    for github.com. On Mar 14, 2026, Paul Romer copied this entry from 
    the github.com documentation at this url:
    url = "https://docs.github.com/en/authentication/"
    url += "keeping-your-account-and-data-secure/"
    url += "githubs-ssh-key-fingerprints"
    """
    if not s:
        s = "github.com ssh-ed25519 "
        s += "AAAAC3NzaC1lZDI1NTE5AAAAIOMqqnkVzrm"
        s += "0SdG6UOoqKLsabgH5C9okWi0dh2l9GKJl"

    found: bool = False
    known_hosts = Path.home() / ".ssh/known_hosts"
    known_hosts_list = known_hosts.read_text().splitlines()
    if known_hosts.exists():
        for line in known_hosts_list:
            if s in line:
                found = True
                break
    else: 
        known_hosts_list = []
    if not found: 
        known_hosts_list.append(s)
        known_hosts.write_text("\n".join(known_hosts_list))
    return 

check_known_hosts()


## Ready to authenticate

At this point, you and the server are in the same position. 

- You have a secret key and the server knows your public key
- The server has a secret key and you know the server's public key

You and the github server can now authenticate each other securely. 

To confirm, run the test in the next cell. It will make an SSH connection to github and reply with your github ID. 

In [11]:
def test_connection():
    """Tests ssh authentication to Github. If you authenticate,  
    you you should get a congratulatory message that contains your 
    github ID. 
    """
    cmd_lst = ["ssh", "-T", "ssh://git@github.com:"]
    cp = run(cmd_lst, capture_output=True, text=True)
    
    github_id = cp.stderr[:cp.stderr.find("!")]
    github_id = github_id[github_id.find(" ")+1:]
    
    if "You've successfully authenticated" in cp.stderr:
        print()
        print(f"    Congratulations {github_id}!")
        print()
        print("    You successfully authenticated with github.com")
        print()

test_connection()


    Congratulations paulmromer!

    You successfully authenticated with github.com



If you don't see a message congratulating you, check all the steps outlined above, including the ones that require that you log in to your github account and paste your public key there. 

## What next? 

At this point, you should be able to get updates of the content in any of the channels in Gennaker. If you want to test this, you can open a terminal. For this channel, the terminal should open inside `~/Gennaker/Channels/DSD/Files`. If you enter `cd ..` you will  move up one level to `~/Gennaker/Channels/DSD`. Next, enter `cd Pubs` to change to the `Pubs` directory. There you can open a terminal and run the command `git status` to see what the current status is for the files in `Pubs`. If you want, you can run `git pull`. If there is any new content on the server, this will bring it down. But there is no need for you to do this. Each time you start Gennaker, it will run the `git pull` command. So with each restart, you'll be up to date. 

Now the SSH authentication is working, there are two directions you can pursue. 

- The is (or there will soon be) a notebook called `challenge_response.ipynb` in the `Pubs` directory of `DSD`. It explains how a key pair lets someone authenticate with someone else.

- There is (or will soon be) a notebook in the same directory called `protecting_ssh.ipynb`. It addresses the question of how you can protect your SSH setup from misuse by malware. As long as no malware is running on your computer, your current setup is safe. But realistically, there is a risk that you will end up with malware on your computer at some point. This notebook explains the trade-offs you face in trying to limit the damage if this happens. It considers in particular the trade-offs associated with the use of passphrases as a way to protect your SSH keys. Don't follow anyone's advice about what to do until you read this explanation. After you do, you'll be able to make a rational decision about the security-convenience trade-offs you will face. 

